# Bias-Variance Tradeoff

In [ ]:
import os
import pandas as pd 
pd.options.display.max_columns = None
import numpy as np 
import statsmodels.formula.api as smf # for OLS regressions
import matplotlib.pyplot as plt

In [ ]:
# Setup:
x = np.arange(1,30,2)
N = 100 # num. sample sets 
P = 1 # highest order of polynomial 
mu = 0; sigma= 3 # normal parameters for epsilon 

## 1. Simulations
Repeat `N` times: 
- Draw $\epsilon \sim N(\mu,\sigma)$
- Generate $y=x+...+x^P+\epsilon$, according to the specified $P$

In [ ]:
def DGP(x,N,P,mu,sigma):
    training = pd.DataFrame()
    log = []
    for i in range(N):
        # print(f'{i}-th sample set:')
        epsilon = np.random.normal(mu,sigma,size=len(x))
        sample = pd.DataFrame({'x':x,'epsilon':epsilon})
        sample['i']=i 
        sample['x2'] = sample['x']**2
        sample['y'] = sample['epsilon'].copy()
        for p in range(1,P+1):
            sample[f'x{int(p)}'] = sample['x']**int(p)
            sample['y'] = sample['y'] + sample[f'x{int(p)}'] 
        # Fit a linear model: 
        model1 =  smf.ols(f"y ~ 1+x", data=sample).fit(cov_type='HC1')
        add = list(model1.params) + [model1.mse_resid.item()]
        sample['ypred1'] = model1.predict()

        # Fit a quadratic model: 
        model2 = smf.ols(f"y ~ 1+x+x2", data=sample).fit(cov_type='HC1')
        sample['ypred2'] = model2.predict()
        add += list(model2.params) + [model2.mse_resid.item()]
        # update data: 
        training = pd.concat([training,sample[['i','x','y','ypred1','ypred2']]], axis=0) 
        # update log: 
        log.append([i]+add)

    # log to dataframe: 
    log = pd.DataFrame(log, columns=['i','a0','a1','mse1','b0','b1','b2','mse2'])
    assert log.shape[0]==N
    # add CEF to data:
    training['cef']=0
    for p in range(1,P+1):
        training[f'x{int(p)}'] = training['x']**int(p)
        training['cef'] = training['cef'] + training[f'x{int(p)}'] 
    return(training, log)


In [ ]:
training1, log1 = DGP(x,N,1,mu, sigma) # x,N,P,mu,sigma

In [ ]:
training2, log2 = DGP(x,N,2,mu, sigma) # x,N,P,mu,sigma

## 2. MSE, Bias, Variance
From `training` - simulated data, compute MSE, bias, variance from linear vs. quadratic model. 

Given $x=x_0$, in the $n$-th sample, let $y_{0(n)}$ denote the actual value of outcome $y$ at $x_0$. And let $\hat{y}_{0(n)}$ denote the predicted value from the linear/quardratic model. We compute the following at $x_0$. 
-  MSE = $\frac{1}{N} \sum_n (y_{0(n)} - \hat{y}_{0(n)})^2$.
-  Bias (squared) = $(f(x_0) - (\frac{1}{N} \sum_n \hat{y}_0))^2$, where CEF $f(x_0)=E[y\vert x=x_0]$ -- does not change across samples!  
-  Variance = $\frac{1}{N} \sum_i (\hat{y}_{0(n)}- (\frac{1}{N} \sum_n \hat{y}_0))^2$. 

In [ ]:
def summary(training):
    training['mse1'] = (training['y']-training['ypred1'])**2
    training['mse2'] = (training['y']-training['ypred2'])**2
    # E[fhat]: 
    E_fhat = training.groupby(['x'])[['ypred1','ypred2']].mean().reset_index(drop=False)
    E_fhat.columns=['x','mean_ypred1','mean_ypred2'] 
    training = pd.merge(training, E_fhat, on=['x'],how='left') 
    # ypred rel. to mean_ypred:
    training['var1'] = (training['ypred1']- training['mean_ypred1'])**2
    training['var2'] = (training['ypred2']- training['mean_ypred2'])**2 


    summary = training.groupby(['x','cef','mean_ypred1','mean_ypred2'])[['mse1','mse2','var1','var2']].mean().reset_index(drop=False)
    summary['bias1'] = (summary['cef']-summary['mean_ypred1'])**2
    summary['bias2'] = (summary['cef']-summary['mean_ypred2'])**2
    return(summary)


### (a) Linear CEF 
$$ f(x) = x $$

In [ ]:
x0= 15 
sum1 = summary(training1)
sum1.loc[sum1['x']==x0,['mse1','var1','bias1','mse2','var2','bias2']] # if want latex: .to_latex(index=False,float_format="%.4f")

### (b) Quadratic CEF
$$ f(x) = x + x^2 $$

In [ ]:
sum2 = summary(training2)
sum2.loc[sum2['x']==x0,['mse1','var1','bias1','mse2','var2','bias2']] # if want latex: .to_latex(index=False,float_format="%.4f")

## 3. Visualization

In [ ]:
def plt_fit(l_samples,training,yvar):
    fig, ax = plt.subplots()
    cmap = plt.cm.tab10.colors
    markers = ['o', 's', '^', 'x', 'D']  
    for k,i in enumerate(l_samples):
        # Scatter for group 0
        ax.scatter(training.loc[training['i']==i, 'x'], training.loc[training['i']==i, yvar], 
                color=cmap[k], marker=markers[k], label=f'Sample {i}', alpha=0.7)
    # 45-degree line
    lo = min(training['x'].min(), training['y'].min())
    hi = max(training['x'].max(), training['y'].max())
    ax.plot([lo, hi], [lo, hi], '--', color='black', label='45° line')

    # Labels and legend
    ax.set_xlabel('x'); ax.set_ylabel('predicted y (quadratic)')
    ax.legend()
    plt.show()

In [ ]:
log1.describe()

In [ ]:
log1[['b1','b2']].corr()

In [ ]:
i1 = log1.loc[log1['b1']<0.5,'i'].sample().item()
i2 = log1.loc[log1['b1']>1.5,'i'].sample().item()
i3 = log1.loc[log1['b2']>0,'i'].sample().item()
print(log1.loc[log1['i'].isin([i1,i2,i3])])


In [ ]:
plt_fit([i1,i2,i3],training1,'ypred2')

In [ ]:
plt_fit([i1,i2,i3],training1,'ypred1')